# Link Prediction -- R-GCN

Trains an `RGCNEncoder` + `DistMultDecoder` on the ~10k-case induced
subgraph built by `link_prediction_setup.ipynb`, using a small validation
search over hidden dimension only (everything else anchored on this
model+schema's already-tuned node-classification hyperparameters from
`rgcn/{schema}_sequential_tuning_results.csv`), then evaluates filtered,
type-constrained ranking (MRR / Hits@1/3/10) per relation on the held-out
test triples.

**Run `link_prediction_setup.ipynb` first.**

In [ ]:
%restart_python

In [ ]:
# Mitigates fragmentation and clears GPU memory a leftover process may hold.
# Must run before any CUDA context is created, right after the restart above.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
import os
import sys

os.environ['PYTORCH_JIT'] = '0'
repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import torch
import pandas as pd

from node_prediction.gnn_tools import cleanup, parse_neighbors
from link_prediction.link_prediction_tools import RGCNEncoder, DistMultDecoder, run_small_lp_validation_search, run_small_lp_secondary_search, quick_sanity_check

In [ ]:
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
LP_DIR = f"{volume}link_prediction/"

FEATURE_DIM = 1024

# Deliberately smaller than Schlichtkrull et al. (2018)'s own LP hidden-dim
# (200-500) -- compute-constrained; this is a supporting experiment, not the
# primary one.
DIM_GRID = [32, 64]

# Run this notebook once per schema, in separate kernel sessions -- see
# link_prediction_setup.ipynb's SCHEMA_LABEL convention.
SCHEMA_LABEL = 'sulo'  # <<< CHANGE TO 'base' FOR THE SECOND RUN

# True resumes from the last checkpoint per hidden_dim instead of
# retraining from scratch (see train_link_prediction_model's save_path/resume).
RESUME_LP_TRAINING = True
CHECKPOINT_DIR = f"{LP_DIR}checkpoints/"

# Caps triples ranked per periodic eval checkpoint (not the final test
# evaluation, which always uses the full set) -- keeps in-training checks cheap.
MAX_VAL_EVAL_TRIPLES = 1000


def get_anchor_hparams(tuning_csv_path):
    """
    Anchors everything except embedding/hidden dimension on the already-tuned
    node-classification winner for this model+schema (peak val_roc_auc among
    Phase 3 verification rows, same selection rule
    RGCN.ipynb/graph_sage.ipynb/analysis.ipynb already use).
    """
    df = pd.read_csv(tuning_csv_path)
    ok = df[df['status'] == 'ok']
    phase3 = ok[ok['phase'].isin(['3_Verification', '3_Verification_Cached'])]
    if len(phase3) == 0:
        phase3 = ok
    best = phase3.loc[phase3['val_roc_auc'].idxmax()]
    bases = best['num_bases']
    bases = None if pd.isna(bases) or bases == 'None' else int(float(bases))
    return {
        'lr': float(best['lr']),
        'weight_decay': float(best['weight_decay']),
        'num_bases': bases,
        'layers': int(best['layers']),
        'neighbors': parse_neighbors(best['neighbors']),
        'epochs': 100,
        'patience': 15,
        'eval_frequency': 5,  # re-embeds every relation's full candidate pool -- too costly every epoch
        # neg_per_pos=4: 1-vs-1 training contrast is easier than the full-pool
        # eval task, so AUC looked strong while MRR stayed near chance at
        # neg_per_pos=1 -- raising it narrows that gap. Raises per-epoch cost
        # by ~(1+neg_per_pos)/2.
        'neg_per_pos': 4,
        # 'macro' (unweighted mean per-relation MRR) instead of 'pooled'
        # (triple-count-weighted, dominated by huge pools like Task) -- keeps
        # early stopping sensitive to smaller, more learnable relations.
        'monitor': 'macro',
    }

## Run the small validation search + final test evaluation

In [ ]:
lp_data = torch.load(f"{LP_DIR}{SCHEMA_LABEL}_lp_data.pt", weights_only=False)
# mp_sub_graph_data (not sub_graph_data): has the disjoint train-supervision
# slice removed so training never sees the edge it's predicting as a neighbor
# (see link_prediction_setup.ipynb's "Mask held-out triples" section).
sub_graph_data = lp_data['mp_sub_graph_data']
# vector_table kept on CPU -- encoders gather from it per-batch and move
# only the small result to GPU (see link_prediction_tools._FeaturizationMixin).
vector_table = lp_data['vector_table']

num_relations = sub_graph_data.edge_type.max().item() + 1  # ENCODER only -- real graph relations, message passing never sees virtual role-chain ids
node_index = sub_graph_data.n_id_to_vec_row
usage_counts = torch.bincount(node_index, minlength=vector_table.shape[0])
categorical_ids = torch.where(usage_counts > 1)[0]

nc_tuning_csv = f"{volume}rgcn/{SCHEMA_LABEL}_sequential_tuning_results.csv"
anchor = get_anchor_hparams(nc_tuning_csv)
print(f"[{SCHEMA_LABEL}] Anchored hyperparameters from NC tuning: {anchor}")

def encoder_factory(dim, _vt=vector_table, _cat=categorical_ids,
                     _nr=num_relations, _bases=anchor['num_bases'], _layers=anchor['layers']):
    return RGCNEncoder(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=[dim] * _layers,
        num_relations=_nr,
        vector_table=_vt,
        categorical_row_ids=_cat,
        num_bases=_bases,
    )

def decoder_factory(dim, _nr=lp_data['decoder_num_relations']):
    return DistMultDecoder(num_relations=_nr, hidden_dim=dim)

final_encoder, final_decoder, test_metrics, search_df, best_dim = run_small_lp_validation_search(
    model_name='RGCN', encoder_factory=encoder_factory, decoder_factory=decoder_factory,
    sub_graph_data=sub_graph_data,
    train_triples_by_relation=lp_data['train_sup_triples_by_relation'],
    val_triples_by_relation=lp_data['val_triples_by_relation'],
    test_triples_by_relation=lp_data['test_triples_by_relation'],
    candidate_pools=lp_data['candidate_pools'],
    true_tails_by_relation=lp_data['true_tails_by_relation'],
    relation_fwd_ids=lp_data['relation_fwd_ids'],
    dim_grid=DIM_GRID,
    fixed_hparams=anchor,
    csv_path=f"{LP_DIR}rgcn_{SCHEMA_LABEL}_lp_dimsearch_tuning_results.csv",
    schema_label=SCHEMA_LABEL,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=RESUME_LP_TRAINING,
    max_val_eval_triples=MAX_VAL_EVAL_TRIPLES,
)

rows = []
for relation_name, m in test_metrics.items():
    # hidden_dim recorded explicitly -- these per-relation results only ever
    # reflect the ONE winning config from the search above, not every dim tried
    row = {'model': 'RGCN', 'schema': SCHEMA_LABEL, 'hidden_dim': best_dim, 'relation': relation_name}
    if relation_name in lp_data['train_sup_triples_by_relation']:
        # num_train_triples reports the SUPERVISED (disjoint) slice -- what
        # actually drove gradient updates -- not the full pre-split train
        # count (see lp_data['train_triples_by_relation'] for that, used by
        # link_prediction_transe.ipynb instead).
        row['num_train_triples'] = int(lp_data['train_sup_triples_by_relation'][relation_name].size(0))
        row['num_val_triples'] = int(lp_data['val_triples_by_relation'][relation_name].size(0))
    row.update(m)
    rows.append(row)
results_df = pd.DataFrame(rows)
dim_results_path = f"{LP_DIR}rgcn_{SCHEMA_LABEL}_lp_dimsearch_results.csv"
results_df.to_csv(dim_results_path, index=False)
print(f"\nSaved (dim search only, superseded by the num_bases/neg_per_pos steps below): {dim_results_path}")
print(results_df.to_string(index=False))

del final_encoder, final_decoder
cleanup()

## Sequential LP tuning extension: num_bases (RGCN only)

Additive on top of the dim search -- `dim` stays locked, only `num_bases`
is swept. GraphSAGE has no bases concept, so this step is RGCN-only.

In [ ]:
# num_bases must be <= num_relations (8 SULO / 38 Base). Brackets
# Schlichtkrull et al. (2018)'s LP setting (B=2), not the NC tuning grid's.
BASES_GRID = [2, 4, 7] if SCHEMA_LABEL == 'sulo' else [5, 10, 20]

def bases_encoder_factory(dim, num_bases, _vt=vector_table, _cat=categorical_ids,
                           _nr=num_relations, _layers=anchor['layers']):
    return RGCNEncoder(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=[dim] * _layers,
        num_relations=_nr,
        vector_table=_vt,
        categorical_row_ids=_cat,
        num_bases=num_bases,
    )

bases_encoder, bases_decoder, bases_test_metrics, bases_search_df, best_bases = run_small_lp_secondary_search(
    model_name='RGCN', param_name='num_bases', param_grid=BASES_GRID, dim=best_dim,
    encoder_factory=bases_encoder_factory, decoder_factory=decoder_factory,
    sub_graph_data=sub_graph_data,
    train_triples_by_relation=lp_data['train_sup_triples_by_relation'],
    val_triples_by_relation=lp_data['val_triples_by_relation'],
    test_triples_by_relation=lp_data['test_triples_by_relation'],
    candidate_pools=lp_data['candidate_pools'],
    true_tails_by_relation=lp_data['true_tails_by_relation'],
    relation_fwd_ids=lp_data['relation_fwd_ids'],
    fixed_hparams=anchor,
    csv_path=f"{LP_DIR}rgcn_{SCHEMA_LABEL}_lp_basessearch_tuning_results.csv",
    schema_label=SCHEMA_LABEL,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=RESUME_LP_TRAINING,
    max_val_eval_triples=MAX_VAL_EVAL_TRIPLES,
    rebuild_encoder=True,
)
print(f"[{SCHEMA_LABEL}] Best num_bases: {best_bases}")

del bases_encoder, bases_decoder
cleanup()

## Sequential LP tuning extension: neg_per_pos

Final step -- `dim`/`num_bases` locked in, only `neg_per_pos` swept.
This step's output is the notebook's canonical results file.

In [ ]:
NEG_PER_POS_GRID = [1, 4, 10]

def final_encoder_factory(dim, _vt=vector_table, _cat=categorical_ids,
                           _nr=num_relations, _bases=best_bases, _layers=anchor['layers']):
    return RGCNEncoder(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=[dim] * _layers,
        num_relations=_nr,
        vector_table=_vt,
        categorical_row_ids=_cat,
        num_bases=_bases,
    )

neg_anchor = dict(anchor)
neg_anchor['num_bases'] = best_bases  # record-keeping only -- applied via final_encoder_factory's closure, not read from this dict

neg_csv_path = f"{LP_DIR}rgcn_{SCHEMA_LABEL}_lp_tuning_results.csv"
final_encoder, final_decoder, test_metrics, neg_search_df, best_neg_per_pos = run_small_lp_secondary_search(
    model_name='RGCN', param_name='neg_per_pos', param_grid=NEG_PER_POS_GRID, dim=best_dim,
    encoder_factory=final_encoder_factory, decoder_factory=decoder_factory,
    sub_graph_data=sub_graph_data,
    train_triples_by_relation=lp_data['train_sup_triples_by_relation'],
    val_triples_by_relation=lp_data['val_triples_by_relation'],
    test_triples_by_relation=lp_data['test_triples_by_relation'],
    candidate_pools=lp_data['candidate_pools'],
    true_tails_by_relation=lp_data['true_tails_by_relation'],
    relation_fwd_ids=lp_data['relation_fwd_ids'],
    fixed_hparams=neg_anchor,
    csv_path=neg_csv_path,
    schema_label=SCHEMA_LABEL,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=RESUME_LP_TRAINING,
    max_val_eval_triples=MAX_VAL_EVAL_TRIPLES,
    rebuild_encoder=False,
)
history_path = neg_csv_path.replace('.csv', '_training_history.csv')
print(f"[{SCHEMA_LABEL}] Best neg_per_pos: {best_neg_per_pos}")

rows = []
for relation_name, m in test_metrics.items():
    # hidden_dim/num_bases/neg_per_pos recorded explicitly -- these
    # per-relation results only ever reflect the ONE winning config from
    # the full sequential search (dim -> num_bases -> neg_per_pos), not
    # every candidate tried at each step
    row = {'model': 'RGCN', 'schema': SCHEMA_LABEL, 'hidden_dim': best_dim,
           'num_bases': best_bases, 'neg_per_pos': best_neg_per_pos, 'relation': relation_name}
    if relation_name in lp_data['train_sup_triples_by_relation']:
        row['num_train_triples'] = int(lp_data['train_sup_triples_by_relation'][relation_name].size(0))
        row['num_val_triples'] = int(lp_data['val_triples_by_relation'][relation_name].size(0))
    row.update(m)
    rows.append(row)
results_df = pd.DataFrame(rows)
results_path = f"{LP_DIR}rgcn_{SCHEMA_LABEL}_lp_results.csv"
results_df.to_csv(results_path, index=False)
print(f"\nSaved final results (dim={best_dim}, num_bases={best_bases}, neg_per_pos={best_neg_per_pos}): {results_path}")
print(results_df.to_string(index=False))

del final_encoder, final_decoder
cleanup()

## Quick sanity check

In [ ]:
# history_path/results_path are set by the neg_per_pos sequential-search
# cell above (the final, canonical output of the dim -> num_bases -> neg_per_pos chain)
_ = quick_sanity_check(results_path, history_path, 'RGCN', SCHEMA_LABEL, 'hidden_dim', best_dim)